In [37]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD

In [38]:
# Load dataset
df = pd.read_excel("movie_ratings_dataset.xlsx")

df.head()

,userId,movieId,title,year,genres,rating,timestamp,director,popularity,vote_count,overview
0,103,23,The Avengers,2012,Action|Adventure|Sci-Fi,3.5,1327113340,Joss Whedon,121.0,38420,Earth's mightiest heroes must come together an...
1,436,16,Titanic,1997,Drama|Romance,4.0,1276035900,James Cameron,95.3,27906,A seventeen-year-old aristocrat falls in love ...
2,861,1,The Shawshank Redemption,1994,Drama,5.0,957402704,Frank Darabont,69.4,24661,Two imprisoned men bond over a number of years...
3,271,12,Schindler's List,1993,Biography|Drama|History,3.5,1430438625,Steven Spielberg,54.8,14405,"In German-occupied Poland during World War II,..."
4,107,24,Iron Man,2008,Action|Adventure|Sci-Fi,2.0,1299251453,Jon Favreau,98.2,27127,"After being held captive in an Afghan cave, bi..."


In [39]:
df.columns

Index(['userId', 'movieId', 'title', 'year', 'genres', 'rating', 'timestamp',
       'director', 'popularity', 'vote_count', 'overview'],
      dtype='object')

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   userId      10000 non-null  int64  
 1   movieId     10000 non-null  int64  
 2   title       10000 non-null  object 
 3   year        10000 non-null  int64  
 4   genres      10000 non-null  object 
 5   rating      10000 non-null  float64
 6   timestamp   10000 non-null  int64  
 7   director    10000 non-null  object 
 8   popularity  10000 non-null  float64
 9   vote_count  10000 non-null  int64  
 10  overview    10000 non-null  object 
dtypes: float64(2), int64(5), object(4)
memory usage: 859.5+ KB


In [41]:
df.describe()

,userId,movieId,year,rating,timestamp,popularity,vote_count
count,10000.000000,10000.000000,10000.000000,10000.000000,1.000000e+04,10000.000000,10000.000000
mean,504.470600,25.354700,2007.325800,3.670450,1.326916e+09,76.309930,20609.258400
std,289.724434,14.355657,11.338585,1.036306,2.183117e+08,18.595475,7908.496909
min,1.000000,1.000000,1972.000000,0.500000,9.467164e+08,43.900000,9379.000000
25%,251.000000,13.000000,1999.000000,3.000000,1.138993e+09,61.800000,14182.000000
50%,506.500000,25.000000,2012.000000,4.000000,1.324542e+09,75.500000,18901.500000
75%,758.000000,38.000000,2015.000000,4.500000,1.519044e+09,86.800000,27138.000000
max,1000.000000,50.000000,2019.000000,5.000000,1.703974e+09,147.500000,42663.000000


In [42]:
movies = df[['title', 'genres', 'overview', 'director']].drop_duplicates()

movies.fillna("", inplace=True)

movies["combined_features"] = (
    movies["genres"] + " " +
    movies["overview"] + " " +
    movies["director"]
)

movies.head()

,title,genres,overview,director,combined_features
0,The Avengers,Action|Adventure|Sci-Fi,Earth's mightiest heroes must come together an...,Joss Whedon,Action|Adventure|Sci-Fi Earth's mightiest hero...
1,Titanic,Drama|Romance,A seventeen-year-old aristocrat falls in love ...,James Cameron,Drama|Romance A seventeen-year-old aristocrat ...
2,The Shawshank Redemption,Drama,Two imprisoned men bond over a number of years...,Frank Darabont,Drama Two imprisoned men bond over a number of...
3,Schindler's List,Biography|Drama|History,"In German-occupied Poland during World War II,...",Steven Spielberg,Biography|Drama|History In German-occupied Pol...
4,Iron Man,Action|Adventure|Sci-Fi,"After being held captive in an Afghan cave, bi...",Jon Favreau,Action|Adventure|Sci-Fi After being held capti...


In [43]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(movies["combined_features"])

cosine_sim = cosine_similarity(tfidf_matrix)

In [25]:
def recommend_content(movie_title, top_n=5):

    idx = movies[movies['title'] == movie_title].index[0]

    similarity_scores = list(enumerate(cosine_sim[idx]))
    similarity_scores = sorted(similarity_scores,
                               key=lambda x: x[1],
                               reverse=True)

    recommendations = []
    for i in similarity_scores[1:top_n+1]:
        recommendations.append(movies.iloc[i[0]]['title'])

    return recommendations


recommend_content("Toy Story")

['Finding Nemo',
 'The Grand Budapest Hotel',
 'A Quiet Place',
 'Black Panther',
 'The Lion King']

In [54]:
svd = TruncatedSVD(n_components=20, random_state=42)

matrix_reduced = svd.fit_transform(user_movie_matrix)

reconstructed_matrix = np.dot(matrix_reduced, svd.components_)

predicted_ratings = pd.DataFrame(reconstructed_matrix,
                                 index=user_movie_matrix.index,
                                 columns=user_movie_matrix.columns)

In [55]:
def recommend_svd(user_id, top_n=5):

    user_row = predicted_ratings.loc[user_id]

    already_rated = user_movie_matrix.loc[user_id] > 0

    recommendations = user_row[~already_rated] \
                        .sort_values(ascending=False) \
                        .head(top_n)

    return recommendations.index.tolist()


recommend_svd(1)

['Parasite', 'Zero Dark Thirty', 'Forrest Gump', 'Knives Out', 'Inception']

In [56]:
movie_stats = df.groupby('title').agg({
    'rating': 'mean',
    'vote_count': 'mean',
    'popularity': 'mean'
}).reset_index()

movie_stats['score'] = (
    movie_stats['rating'] * 0.6 +
    movie_stats['popularity'] * 0.3 +
    movie_stats['vote_count'] * 0.1
)

movie_stats.sort_values(by='score', ascending=False).head()

,title,rating,vote_count,popularity,score
4,Avengers: Endgame,3.595455,42166.995455,142.540455,4261.618955
34,The Avengers,3.563679,38742.566038,118.300943,3911.885094
18,Inception,3.640625,34487.354167,88.960938,3477.608073
6,Black Panther,3.624324,33250.275676,108.118378,3359.637676
35,The Dark Knight,3.618687,31271.919192,95.290404,3157.950253


In [60]:
def recommend_by_genre(genre_name, top_n=5):

    # Filter movies by genre (case insensitive)
    filtered = df[df['genres'].str.contains(genre_name,
                                            case=False,
                                            na=False)]

    # Group by movie & calculate average rating
    movie_scores = filtered.groupby('title').agg({
        'rating': 'mean',
        'popularity': 'mean',
        'vote_count': 'mean'
    }).reset_index()

    # Create weighted score
    movie_scores['score'] = (
        movie_scores['rating'] * 0.5 +
        movie_scores['popularity'] * 0.3 +
        movie_scores['vote_count'] * 0.2
    )

    movie_scores = movie_scores.sort_values(by='score',
                                            ascending=False)

    return movie_scores['title'].head(top_n).tolist()

In [61]:
print("🎬 Live Movie Recommender")

keyword = input("What type of movie do you want? (Love, Action, Horror, etc): ")

print("Recommended Movies:")
print(recommend_by_keyword(keyword))

🎬 Live Movie Recommender
What type of movie do you want? (Love, Action, Horror, etc): Horror
Recommended Movies:
['A Quiet Place', 'Get Out', 'Hereditary', 'Midsommar']
